<a href="https://colab.research.google.com/github/alee52/LLM_AgenticAI/blob/main/eval_fine_tuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade bitsandbytes trl

In [ ]:

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
from util import evaluate

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "categorize_products_no_cate"
HF_USER = "leearum95" # your HF name here!

LITE_MODE = False

DATA_USER = "leearum95"
DATASET_NAME = f"{DATA_USER}/items_prompts_full_no_category"
if LITE_MODE:
  # RUN_NAME = "2026-05-07_19.00.38-lite"
  REVISION = None
else:
  RUN_NAME = "2026-05-08_18.44.24"
  REVISION = None


PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8use_bf16 = capability[0] >= 8

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, min_new_tokens = 2,max_new_tokens=10)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [ ]:
train = dataset['train']
# model_predict(train[10])

# print(train['prompt'])

In [ ]:
model_predict(train[1000])
# print(test[0]['prompt'])
# print(test[0]['completion'])


In [ ]:

print(train[100]['completion'])

In [ ]:
print(test[2500]['prompt'])

In [ ]:
def model_predict2(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            min_new_tokens=2,
            max_new_tokens=10,
            return_dict_in_generate=True,
            output_scores=True,
        )

    output_ids = outputs.sequences
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    print("Generated text:")
    print(repr(tokenizer.decode(generated_ids)))

    print("\nTop 5 predicted tokens at each generated step:")

    for step, logits in enumerate(outputs.scores):
        # logits shape: [batch_size, vocab_size]
        probs = torch.softmax(logits[0], dim=-1)

        top_probs, top_token_ids = torch.topk(probs, k=5)

        chosen_token_id = generated_ids[step].item()
        chosen_text = tokenizer.decode([chosen_token_id])

        print(f"\nStep {step + 1}")
        print(f"Chosen token: {chosen_token_id} {repr(chosen_text)}")

        for prob, token_id in zip(top_probs, top_token_ids):
            token_id = token_id.item()
            token_text = tokenizer.decode([token_id])
            print(f"{token_id:>8} {repr(token_text):>15} prob={prob.item():.6f}")

    return tokenizer.decode(generated_ids)

In [ ]:
print(test[1200]['prompt'])
print(test[1200]['completion'])
print(tokenizer.encode(test[1200]['completion']))
model_predict2(test[1200])